# 00 · Extract tile and whole-FOV traces

Run this once per recording, before the other notebooks. It reads each Suite2p `planeN/ops.npy` and `data.bin` and writes, under the recording's `export_root`:
- **whole-FOV raw F**: the mean over the valid registered crop, before any ΔF/F
- **raw F for each tile**, after the brightness screen

Recordings whose `export_root` is their Suite2p folder already have exports from the old notebooks and are skipped. Nothing is ever written into a Suite2p folder on the share. Writing exports into a Suite2p detection folder is refused, because the tile `F.npy` would overwrite Suite2p's ROI `F.npy`.

The baseline set here (`baseline_window_s`, `baseline_percentile`) only decides the ΔF/F saved alongside the raw F, which the old event/pupil scripts read. Notebooks 01–05 recompute every baseline definition from the raw F.

Averaging a ~9 GB binary over the network takes several minutes per plane. Plane folders that already have an export are skipped, so delete an export, or point `export_root` at a new folder, to redo it with new settings.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from nlight import plotting, session, suite2p_io, traces

plotting.use_lab_style()
RECORDINGS = session.load_recordings(Path('../../configs/nlight_recordings.json'))
print('Available:', list(RECORDINGS))

Available: ['corridor2 passive viewing', 'corridor2 baseline', 'passive (4 planes)', 'no puff (2x, 4 planes)', 'puff (2x, 4 planes)']


In [2]:
SETTINGS = traces.TraceSettings(tile_size_px=32, exclude_below_percentile=0,
                                baseline_window_s=60, baseline_percentile=50)
TO_EXTRACT = ['corridor2 passive viewing', 'corridor2 baseline']

In [3]:
for name in TO_EXTRACT:
    rec = RECORDINGS[name]
    if rec.export_root == rec.suite2p_root:
        print(f'{name}: exports live in its Suite2p folder; skipped (read only)')
        continue
    for plane_dir in suite2p_io.plane_dirs(rec.suite2p_root):
        done = (rec.export_root / plane_dir.name / 'whole_image_roi_dff.npz').exists() and \
               (rec.export_root / plane_dir.name / 'tile_fluorescence_dff.npz').exists()
        if done:
            print(f'{name} {plane_dir.name}: already extracted')
            continue
        print(f'{name} {plane_dir.name}: extracting …', flush=True)
        traces.extract_tiles(plane_dir, SETTINGS, out_root=rec.export_root)
        traces.extract_whole_fov(plane_dir, SETTINGS, out_root=rec.export_root)

corridor2 passive viewing plane0: already extracted
corridor2 baseline plane0: already extracted
